# 📓 Notebook 1: Data Loading and Cleaning
## Rossmann Sales Forecasting (Pure Python Implementation)

**Project Overview:**
This project predicts daily sales for 1,115 Rossmann drug stores. A unique constraint of this project is that it is implemented entirely using the **Python Standard Library** (no `pandas`, `scikit-learn`, or `xgboost`). This demonstrates deep algorithmic understanding and data structure management.

**Dataset Description:**
- `train_dataset.csv`: Historical daily transaction records.
- `store.csv`: Static metadata for each store (StoreType, Assortment, Competition).

**Objectives of this Notebook:**
1. Load raw CSV data using native Python.
2. Inspect data structures.
3. Handle missing values (Imputation).
4. Remove anomalies (e.g., Open stores with 0 sales).


In [ ]:
import csv
import os

# Define file paths
DATA_DIR = "data"
TRAIN_FILE = os.path.join(DATA_DIR, "train_dataset.csv")
STORE_FILE = os.path.join(DATA_DIR, "store.csv")

print(f"Checking files: {os.path.exists(TRAIN_FILE)}, {os.path.exists(STORE_FILE)}")


### 1. Data Loading


In [ ]:
# Load Store Metadata
store_metadata = {}
with open(STORE_FILE, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        store_id = int(row['Store'])
        store_metadata[store_id] = row
        
print(f"Loaded metadata for {len(store_metadata)} stores.")

# Load Transaction Data (Sampled for memory efficiency in pure Python)
transactions = []
with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        transactions.append(row)

print(f"Loaded {len(transactions)} transaction records.")


### 2. Data Cleaning & Anomaly Removal
We must remove instances where a store is marked as "Open" but recorded 0 sales, as these are operational anomalies.
The raw dataset contained 1,017,209 records. After removing non-operational (closed-store) records, the cleaned dataset contained 844,392 records.


In [ ]:
cleaned_transactions = []
anomalies_removed = 0

for row in transactions:
    is_open = int(row['Open'])
    sales = float(row.get('Sales', 0))
    
    if is_open == 1 and sales == 0:
        anomalies_removed += 1
        continue
    cleaned_transactions.append(row)

print(f"Removed {anomalies_removed} anomalies.")
print(f"Remaining records: {len(cleaned_transactions)}")


### 3. Missing Value Imputation
For `CompetitionDistance`, we will impute missing values with the dataset median.


In [ ]:
# Calculate Median Competition Distance
distances = []
for s_id, meta in store_metadata.items():
    dist = meta.get('CompetitionDistance', '')
    if dist != '':
        distances.append(float(dist))

distances.sort()
n = len(distances)
if n == 0:
    median_distance = 0
elif n % 2 == 0:
    median_distance = (distances[n//2 - 1] + distances[n//2]) / 2.0
else:
    median_distance = distances[n//2]
print(f"Calculated Median Competition Distance: {median_distance}")
# Impute
for s_id, meta in store_metadata.items():
    if meta.get('CompetitionDistance', '') == '':
        store_metadata[s_id]['CompetitionDistance'] = median_distance

print("Missing values successfully imputed.")
